## clean phase 1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(
    r"../data/raw/arabic/Arabic_tweets_sentiment.csv",
    sep="\t"
)

print(df.shape)

(2286128, 4)


In [3]:
df = df[["text", "class"]].copy()

In [7]:
# Keep only required columns
df_model = df[["text", "class"]].copy()

# Rename class -> sentiment
df_model.rename(columns={"class": "sentiment"}, inplace=True)

# Convert labels to lowercase
df_model["sentiment"] = df_model["sentiment"].str.lower()

df_model.head()

,text,sentiment
0,انت فارقني انا تمام الشعب الاردني شعبي وجزء من...,negative
1,انت ماقابلته الاردن رحت مقابلو مصر عار عليكم,negative
2,بعد طلبه فتح الحدود ومحاربة الاسرائيليين ممثل ...,neutral
3,وناااسه راح تصير اعماركم؟ ستشاهده خبر عظيم جد...,neutral
4,ممثل شهير يكشف مأساة عائلة زوجته غزة عمليهطوفا...,negative


In [8]:
df_model["sentiment"].value_counts()

sentiment
negative    1148870
neutral      613142
positive     524116
Name: count, dtype: int64

In [9]:
df_model["sentiment"] = df_model["sentiment"].str.lower()

In [10]:
df = df_model.dropna(subset=["text"])

df_model.reset_index(drop=True, inplace=True)

In [11]:
df_model["text"] = df_model["text"].astype(str).str.strip()

In [12]:
df_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 2286128 entries, 0 to 2286127
Data columns (total 2 columns):
 #   Column     Dtype
---  ------     -----
 0   text       str  
 1   sentiment  str  
dtypes: str(2)
memory usage: 474.5 MB


In [13]:
df_model.isnull().sum()

text         5612
sentiment       0
dtype: int64

In [14]:
df_model.duplicated().sum()

np.int64(1092501)

In [15]:
df_model["sentiment"].value_counts()

sentiment
negative    1148870
neutral      613142
positive     524116
Name: count, dtype: int64

## Cleaning

In [18]:
import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Decode HTML entities
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove mentions
    text = re.sub(r"@\w+", "", text)

    # Remove RT
    text = re.sub(r"^RT\s+", "", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [19]:
df_model["text"] = df_model["text"].apply(clean_text)

In [20]:
# Remove empty texts after cleaning
df_model = df_model[df_model["text"].str.strip() != ""]
df_model.reset_index(drop=True, inplace=True)

In [21]:
df_model = df_model.drop_duplicates(subset=["text"])

In [22]:
df_model.head()

,text,sentiment
0,انت فارقني انا تمام الشعب الاردني شعبي وجزء من...,negative
1,انت ماقابلته الاردن رحت مقابلو مصر عار عليكم,negative
2,بعد طلبه فتح الحدود ومحاربة الاسرائيليين ممثل ...,neutral
3,وناااسه راح تصير اعماركم؟ ستشاهده خبر عظيم جدا...,neutral
4,ممثل شهير يكشف مأساة عائلة زوجته غزة عمليهطوفا...,negative


In [23]:
# Remove missing texts
df_model = df_model.dropna(subset=["text"])

# Remove empty texts
df_model = df_model[df_model["text"].str.strip() != ""]

df_model.reset_index(drop=True, inplace=True)

## Sampling
Since dataset is larg of 2M smaple, we'll take only 240K.
80K per label.

In [24]:
N = 80000

In [25]:
negative = df_model[df_model["sentiment"] == "negative"].sample(
    N,
    random_state=42
)

neutral = df_model[df_model["sentiment"] == "neutral"].sample(
    N,
    random_state=42
)

positive = df_model[df_model["sentiment"] == "positive"].sample(
    N,
    random_state=42
)

In [26]:
final_df = pd.concat(
    [
        negative,
        neutral,
        positive
    ],
    ignore_index=True
)

In [27]:
final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [28]:
final_df["sentiment"].value_counts()

sentiment
positive    80000
neutral     80000
negative    80000
Name: count, dtype: int64

In [29]:
final_df.to_csv(
    "../data/processed/arabic_final_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

## Split data
Train = 80%
Validation = 10%
Test = 10%

In [31]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    final_df,
    test_size=0.2,
    stratify=final_df["sentiment"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["sentiment"],
    random_state=42
)

In [32]:
print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(192000, 2)
(24000, 2)
(24000, 2)


In [33]:
print(train_df.isnull().sum())
print(valid_df.isnull().sum())
print(test_df.isnull().sum())

text         0
sentiment    0
dtype: int64
text         0
sentiment    0
dtype: int64
text         0
sentiment    0
dtype: int64


In [44]:
train_df.to_csv("../data/processed/arabic/train.csv", index=False, encoding="utf-8-sig")

valid_df.to_csv("../data/processed/arabic/validation.csv", index=False, encoding="utf-8-sig")

test_df.to_csv("../data/processed/arabic/test.csv", index=False, encoding="utf-8-sig")

In [45]:
train = pd.read_csv(
    "../data/processed/arabic/train.csv",
    keep_default_na=False
)

print(train.isnull().sum())

text         0
sentiment    0
dtype: int64


In [46]:
train = pd.read_csv("../data/processed/arabic/train.csv")
valid = pd.read_csv("../data/processed/arabic/validation.csv")
test = pd.read_csv("../data/processed/arabic/test.csv")

In [48]:
print("========== TRAIN ==========")
print(train.head())

print("\n========== VALIDATION ==========")
print(valid.head())

print("\n========== TEST ==========")
print(test.head())

========== TRAIN ==========
                                                text sentiment
0  الاردن تعترض صاروخ العراق بيجي واحد راسوا مستط...  negative
1                           بتدخلون التاريخ شاء الله  positive
2  إدخالالوقودلغزة إمدادغزةبالماء غزةتباد غزةتحتا...  negative
3  لعنكم الله لعنة تغشاكم انتم والدب الداشر الخاي...  negative
4                    الفقرة حلوه مشعلالقحطاني البزنس   neutral

========== VALIDATION ==========
                                                text sentiment
0  اللهم إنّا نستودعك فلسطين وغزة وأهلها، أمنها و...  negative
1                                يسلمووو قلبي وروحي🤍  positive
2  والله جعلتني اتغزز منها ومن ابوها وجدها ومبين ...  negative
3  كون هذه المعلومه ماخوذه القناه العبريه هذا تصر...   neutral
4          صلوات ربي وازكى سلام عليك حبيبي رسول الله  positive

========== TEST ==========
                                                text sentiment
0  القناة العبرية إطلاق صواريخ غزة نحو الجليل وال...  negative
1  آخر شيء كان يخشاه الشيخ ص

In [49]:
print(train_df.isnull().sum())
print(valid_df.isnull().sum())
print(test_df.isnull().sum())

text         0
sentiment    0
dtype: int64
text         0
sentiment    0
dtype: int64
text         0
sentiment    0
dtype: int64


In [50]:
train = pd.read_csv(
    "../data/processed/arabic/train.csv",
    keep_default_na=False
)

print(train.isnull().sum())

text         0
sentiment    0
dtype: int64


In [51]:
print(df_model.isnull().sum())

text         0
sentiment    0
dtype: int64


In [52]:
df_model[df_model["text"].isnull()].head()

,text,sentiment
